In [1]:
import requests
import pandas as pd

Connexion à l'API

In [1]:
import requests
import os
from dotenv import load_dotenv

# 1. Charger le coffre-fort (.env)
load_dotenv()

# 2. Récupérer la clé secrète
API_KEY = os.getenv('OPENWEATHER_API_KEY')

CITY = 'Casablanca'
URL = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

response = requests.get(URL)

if response.status_code == 200:
    data = response.json()
    print("✅ Données récupérées avec succès")
else:
    print("❌ Erreur lors de la requête API :", response.status_code)

✅ Données récupérées avec succès


Extraction des données

In [2]:
weather_data = {
    'Ville': data['name'],
    'Temperature': data['main']['temp'],
    'Temp_min': data['main']['temp_min'],
    'Temp_max': data['main']['temp_max'],
    'Pression': data['main']['pressure'],
    'Humidité': data['main']['humidity'],
    'Description': data['weather'][0]['description'],
    'Vent_Vitesse':data['wind']['speed']
}
print(weather_data)

{'Ville': 'Casablanca', 'Temperature': 19.07, 'Temp_min': 19.07, 'Temp_max': 19.19, 'Pression': 1019, 'Humidité': 72, 'Description': 'broken clouds', 'Vent_Vitesse': 3.09}


Sauvegarde dans un fichier CSV

In [4]:
import pandas as pd

df = pd.DataFrame([weather_data])
df.to_csv(data['name']+'_weather.csv', index=False)
print("fichier CSV cree avec succees")
df

fichier CSV cree avec succees


,Ville,Temperature,Temp_min,Temp_max,Pression,Humidité,Description,Vent_Vitesse
0,Casablanca,27.02,26.85,29.12,1017,30,clear sky,1.54


Meteo de plusieurs villes

In [5]:
cities=['Casablanca','Manchester', 'Berlin', 'Tokyo', 'New York']
all_data=[]
for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    rep = requests.get(url)
    if rep.status_code == 200:
        dt = rep.json()
        all_data.append({
            'Ville': dt['name'],
            'Temperature': dt['main']['temp'],
            'Humidité': dt['main']['humidity'],
            'Description': dt['weather'][0]['description'],
        })
datas = pd.DataFrame(all_data)
datas.to_csv('cities_weather.csv', index=False)
print("fichier CSV pour les villes cree avec succees")
datas

fichier CSV pour les villes cree avec succees


,Ville,Temperature,Humidité,Description
0,Casablanca,27.02,30,clear sky
1,Manchester,7.34,70,light rain
2,Berlin,17.21,63,few clouds
3,Tokyo,19.57,72,broken clouds
4,New York,7.09,96,mist


In [6]:
pip install schedule

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import schedule
import time
from datetime import datetime

def collect_weather() :
    response = requests.get(URL)
    if response.status_code == 200 :
        data = response.json()
        weather_data = {
            'Datetime' : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'Ville' : data['main'],
            'Temperature' : data['main']['temp'],
            'Humidité' : data['main']['humidity'],
            'Description' : data['weather'][0]['description']
        }
        df = pd.DataFrame([weather_data])
        df.to_csv('meteo_automatique.csv', mode='a', header=not pd.io.common.file_exists('meteo_automatique.csv'), index=False)
        print(f"Donnees collectees a {weather_data['Datetime']}")
schedule.every(1).minutes.do(collect_weather)
print("lancement de la collecte horaire ...")
while True:
    schedule.run_pending()
    time.sleep(1)

lancement de la collecte horaire ...
Donnees collectees a 2026-04-05 11:35:23
Donnees collectees a 2026-04-05 11:36:25


Partie 2 : Collecte depuis une BD relationnelle

In [2]:
%pip install sqlalchemy psycopg2-binary

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 2.1 MB/s eta 0:00:01
   ------------------- -------------------- 1.0/2.1 MB 2.4 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 2.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 2.6 MB/s eta 0:00:01
   ------------------ --------------------- 1.3/2.8 MB 2.7 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.8 MB 2.5 MB/s eta 0:00:01
   --------------------------------- ------ 2.4/2.8 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 2.5 MB/s  0:00:01
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   ---------------------------------------- 0

1. Connexion à la base de données 


In [1]:
from sqlalchemy import create_engine
import pandas as pd

USER = 'postgres'
PASSWORD = 'admin'
HOST = 'localhost'
PORT = '5432'
DB = 'transport_meteo'
engine = create_engine(f'postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}')

2. Extraction des données jointes 


In [13]:
query = '''
SELECT tb.date , tb.ligne , tb.arret , tb.nombre_passagers , m.temperature ,m.pluie , m.vent , m.description
FROM trafic_bus tb
JOIN meteo m ON tb.date = m.date
ORDER BY tb.date , tb.ligne ;
'''
df=pd.read_sql(query,engine)
df

,date,ligne,arret,nombre_passagers,temperature,pluie,vent,description
0,2025-03-20,1,Place Centrale,320,18.5,False,10.0,ensoleillé
1,2025-03-20,1,Place Centrale,320,18.5,False,10.0,ensoleillé
2,2025-03-20,2,Gare Sud,150,18.5,False,10.0,ensoleillé
3,2025-03-20,2,Gare Sud,150,18.5,False,10.0,ensoleillé
4,2025-03-21,1,Place Centrale,210,16.2,True,25.0,pluie
5,2025-03-21,1,Place Centrale,210,16.2,True,25.0,pluie
6,2025-03-21,2,Gare Sud,90,16.2,True,25.0,pluie
7,2025-03-21,2,Gare Sud,90,16.2,True,25.0,pluie
8,2025-03-22,1,Place Centrale,280,17.0,False,15.0,nuageux
9,2025-03-22,1,Place Centrale,280,17.0,False,15.0,nuageux


3. Export CSV 


In [15]:
df.to_csv('trafic_meteo.csv',index=False)
print("Export reussi des donnees dans trafic_meteo.csv")

Export reussi des donnees dans trafic_meteo.csv


4. Automatisation toutes les heures 


In [ ]:
import schedule
import time
from datetime import datetime 

def extract_data():
  df = pd.read_sql(query, engine)
  filename = f"transport_meteo{datetime.now().strftime('%Y-%m-%d_%H_%M_%S')}.csv"
  df.to_csv(filename, index=False)
  print(f"donnees importes avec succes dans {filename}")

schedule.every().hour.do(extract_data)
print("Extraction planifiée toutes les heures...")
while True:
    schedule.run_pending()
    time.sleep(1)


Extraction planifiée toutes les minutes...
donnees importes avec succes dans transport_meteo2026-04-05_20_36_40.csv
donnees importes avec succes dans transport_meteo2026-04-05_20_37_40.csv


KeyboardInterrupt: 